# 0. Redis 안을 들여다보기

Supabase에는 **Table Editor**가 있어서 저장한 데이터를 눈으로 볼 수 있었습니다.
Redis에는 그런 화면이 기본으로 없습니다. 대신 두 가지 방법을 씁니다.

| 방법 | 언제 |
| --- | --- |
| Redis Cloud 콘솔의 **Data Browser** | 클릭으로 훑어볼 때 |
| **이 노트북의 `show_all()`** | 실습 중 빠르게 확인할 때 |

이 노트북은 실습이 아니라 **도구**입니다. `TODO`가 없습니다.
다른 실습을 하다가 "지금 뭐가 들어 있지?" 싶을 때 열어서 실행합니다.

## 방법 1 — Redis Cloud 콘솔에서 보기

Supabase의 Table Editor에 해당하는 화면입니다.

1. [app.redislabs.com](https://app.redislabs.com) 로그인
2. 만들어둔 데이터베이스 클릭
3. 위쪽 탭에서 **Data Browser** 클릭 (콘솔에 따라 **RedisInsight**로 표시됩니다)

왼쪽에 키 목록이 나오고, 키를 클릭하면 오른쪽에 값·종류·TTL이 보입니다.
검색창에 `day14:*`를 넣으면 실습 키만 걸러집니다.

> **주의:** 키가 바로 안 보이면 새로고침 버튼을 누릅니다. 콘솔은 자동으로 갱신되지 않습니다.

**콘솔이 편한 경우**: 키 이름을 모를 때, 값의 구조를 눈으로 확인할 때
**코드가 편한 경우**: 실습 도중 반복해서 확인할 때, 여러 키를 한 번에 볼 때

## 방법 2 — 코드로 보기

Redis는 **종류마다 값을 꺼내는 명령이 다릅니다.** `get` 하나로는 안 됩니다.

| 종류 | 꺼내는 명령 |
| --- | --- |
| `string` | `r.get(키)` |
| `hash` | `r.hgetall(키)` |
| `list` | `r.lrange(키, 0, -1)` |
| `set` | `r.smembers(키)` |
| `zset` | `r.zrevrange(키, 0, -1, withscores=True)` |

`string`이 아닌 키에 `get`을 쓰면 `WRONGTYPE` 오류가 납니다.
그래서 먼저 `r.type(키)`로 종류를 물어보고, 그에 맞는 명령을 고릅니다.

In [1]:
from redis_client import r

print("ping        :", r.ping())
print("전체 키 개수:", r.dbsize())

ping        : True
전체 키 개수: 0


In [2]:
# 종류별로 값을 꺼내는 방법을 미리 짝지어 둔다.
READERS = {
    "string": lambda key: r.get(key),
    "hash":   lambda key: r.hgetall(key),
    "list":   lambda key: r.lrange(key, 0, -1),
    "set":    lambda key: sorted(r.smembers(key)),
    "zset":   lambda key: r.zrevrange(key, 0, -1, withscores=True),
}

### `show_all()` — 조건에 맞는 키를 전부 보기

`scan_iter`로 키를 찾고, 각각의 종류·TTL·값을 표로 출력합니다.

In [5]:
def show_all(pattern="*"):
    keys = sorted(r.scan_iter(pattern)) ## 전부 가져와

    if not keys:
        print(f"'{pattern}' 에 맞는 키가 없다.")
        return

    print(f"{'키':<26} {'종류':<8} {'TTL':>6}  값")
    print("-" * 88)

    for key in keys:
        kind = r.type(key)
        ttl = r.ttl(key)
        # -1 은 만료 없음, -2 는 키 없음(조회하는 사이 만료된 경우)
        ttl_text = "없음" if ttl == -1 else ("사라짐" if ttl == -2 else f"{ttl}s")
        print(f"{key:<26} {kind:<8} {ttl_text:>6}  {READERS[kind](key)}")

    print()
    print(f"총 {len(keys)}개")

확인해봅니다. 실습을 한 번도 안 돌렸다면 비어 있는 게 정상입니다.

In [7]:
show_all("day14:*")

키                          종류          TTL  값
----------------------------------------------------------------------------------------
day14:chat                 list         없음  ['안녕하세요', '무엇을 도와드릴까요?']
day14:greeting             string       없음  안녕하세요
day14:online               set          없음  ['user1', 'user2']
day14:ranking              zset         없음  [('이영희', 500.0), ('김철수', 300.0)]
day14:user                 hash         없음  {'name': '김철수', 'point': '150'}

총 5개


### 시험 삼아 넣어보기

다섯 종류를 하나씩 넣고 다시 봅니다. `day14:temp`에만 TTL을 걸어둡니다.

In [8]:
r.set("day14:greeting", "안녕하세요")
r.set("day14:temp", "30초 뒤 사라짐", ex=30)
r.hset("day14:user", mapping={"name": "김철수", "point": "150"})
r.rpush("day14:chat", "안녕하세요", "무엇을 도와드릴까요?")
r.sadd("day14:online", "user1", "user2")
r.zadd("day14:ranking", {"이영희": 500, "김철수": 300})

show_all("day14:*")

키                          종류          TTL  값
----------------------------------------------------------------------------------------
day14:chat                 list         없음  ['안녕하세요', '무엇을 도와드릴까요?', '안녕하세요', '무엇을 도와드릴까요?']
day14:greeting             string       없음  안녕하세요
day14:online               set          없음  ['user1', 'user2']
day14:ranking              zset         없음  [('이영희', 500.0), ('김철수', 300.0)]
day14:temp                 string      30s  30초 뒤 사라짐
day14:user                 hash         없음  {'name': '김철수', 'point': '150'}

총 6개


`day14:temp`의 TTL만 숫자로 줄어듭니다. 셀을 다시 실행하면 값이 작아지고,
30초가 지나면 **그 줄 자체가 사라집니다.**

Redis Cloud 콘솔에서도 같은 키들이 보이는지 확인해보세요. 같은 데이터입니다.

### `show_one()` — 키 하나를 자세히

In [ ]:
def show_one(key):
    if not r.exists(key):
        print(f"'{key}' 라는 키가 없다.")
        return

    kind = r.type(key)
    ttl = r.ttl(key)

    print("키   :", key)
    print("종류 :", kind)
    print("TTL  :", "없음" if ttl == -1 else f"{ttl}초 남음")
    print("값   :", READERS[kind](key))


show_one("day14:user")
print()
show_one("day14:없는키")

## `keys()` 와 `scan_iter()` 의 차이

둘 다 같은 결과를 줍니다. 실습에서는 키가 몇 개뿐이라 차이가 없습니다.

| | 동작 | 문제 |
| --- | --- | --- |
| `r.keys("*")` | 한 번에 전부 찾는다 | 키가 많으면 **그동안 Redis 전체가 멈춘다** |
| `r.scan_iter("*")` | 조금씩 나눠 찾는다 | 없음 (조금 느릴 뿐) |

**운영 중인 서버에서 `keys("*")`를 실행하면 서비스가 잠시 멎습니다.**
습관을 처음부터 `scan_iter`로 들입니다.

In [ ]:
print("keys      :", len(r.keys("day14:*")), "개")
print("scan_iter :", len(list(r.scan_iter("day14:*"))), "개")

## 정리

`day14:` 로 시작하는 키만 지웁니다. 다른 데이터는 건드리지 않습니다.

In [ ]:
def clear(pattern="day14:*"):
    keys = list(r.scan_iter(pattern))
    for key in keys:
        r.delete(key)
    print(f"{len(keys)}개 삭제. 남은 키 개수: {r.dbsize()}")


clear()